# WANDS: field-weighted BM25 vs. GTE-small

Compare two methods using only **product name, category and description**.

- **BM25:** separate field indexes, weighted **name=1, category=0.5, description=0.25**;
  regex + Porter stemming, Lucene BM25 (`k1=1.2`, `b=0.75`).
- **Vector search:** concatenate those three fields, generate **one embedding per product** with
  [`thenlper/gte-small`](https://huggingface.co/thenlper/gte-small), and search the full catalog by cosine similarity.
  GTE-small uses raw text without E5-style prefixes. No field-specific vector weighting is applied.

The fixed BM25 weights come from the three-field result in the
[field-weighting post](https://enisteper1.github.io/posts/field-weighting.html).
There is no weight tuning or cross-validation here. Because those weights were previously selected on WANDS,
this is a descriptive comparison on reused queries, not a new held-out estimate.

## 1. Setup and data

Run from the repository root or `retrieval/`. Model downloads require internet on first use;
encoding is local. Both methods use all products and the same evaluable queries.

In [1]:
# Install in the active kernel if needed, then restart it:
# %pip install bm25s numpy pandas nltk sentence-transformers
from pathlib import Path
from zipfile import ZipFile
from functools import lru_cache
import re
import numpy as np
import pandas as pd
import bm25s
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / "retrieval").is_dir() else Path.cwd().parent
FIELDS = ["product_name", "category hierarchy", "product_description"]
WEIGHTS = {"product_name": 1.0, "category hierarchy": 0.5, "product_description": 0.25}
MODEL_NAME, K = "thenlper/gte-small", 100
DEVICE = None  # auto-select; set 'cpu', 'mps' or 'cuda' if needed


def load_table(name):
    path = ROOT / "datasets/wands" / name
    if path.exists():
        return pd.read_csv(path, sep="\t")
    with ZipFile(ROOT / "datasets.zip") as z, z.open(f"datasets/wands/{name}") as f:
        return pd.read_csv(f, sep="\t")


products, queries, labels = [load_table(f"{name}.csv") for name in ["product", "query", "label"]]
products[FIELDS] = products[FIELDS].fillna("").astype(str)
assert products.product_id.is_unique and queries.query_id.is_unique and len(products) >= K
positions = {str(pid): i for i, pid in enumerate(products.product_id)}
labels = labels.assign(qid=labels.query_id.astype(str),
                       doc=labels.product_id.astype(str).map(positions),
                       grade=labels.label.map({"Exact": 2, "Partial": 1, "Irrelevant": 0}))
assert labels[["doc", "grade"]].notna().all().all()
labels = labels.groupby(["qid", "doc"], as_index=False).grade.max()
qrels = {qid: dict(zip(g.doc.astype(int), g.grade.astype(int))) for qid, g in labels.groupby("qid")}
queries = queries[queries.query_id.astype(str).map(
    lambda qid: any(v > 0 for v in qrels.get(qid, {}).values()))].reset_index(drop=True)
grades = [qrels[str(qid)] for qid in queries.query_id]
print(f"{len(products):,} products; {len(queries)} queries; weights: {WEIGHTS}")

42,994 products; 479 queries; weights: {'product_name': 1.0, 'category hierarchy': 0.5, 'product_description': 0.25}


## 2. Common ranking and evaluation

Both methods return up to 100 products, with product-row order breaking ties. BM25 omits zero-score
products; cosine search keeps zero and negative scores. Exact search uses NumPy, avoiding ANN recall loss.

**nDCG@10:** Exact=2, Partial=1, gain `2**grade - 1`. **MRR@10:** first Exact hit, excluding queries
without any Exact label. **Recall@10/100:** Exact + Partial. Unjudged products receive zero gain in
metrics, but are displayed as Unjudged rather than Irrelevant in examples.

In [2]:
def topk(scores, positive_only=False):
    cutoff = np.partition(scores, len(scores) - K)[len(scores) - K]
    ids = np.concatenate([np.flatnonzero(scores > cutoff),
                          np.flatnonzero(scores == cutoff)[:K - np.sum(scores > cutoff)]])
    ids = ids[np.lexsort((ids, -scores[ids]))]
    if positive_only:
        ids = ids[scores[ids] > 0]
    return np.pad(ids, (0, K - len(ids)), constant_values=-1)


def evaluate(ranked):
    values = np.array([[rel.get(int(d), 0) for d in row] for row, rel in zip(ranked, grades)])
    discounts = 1 / np.log2(np.arange(2, 12))
    ideal = [sorted(rel.values(), reverse=True)[:10] for rel in grades]
    idcg = np.array([np.dot(2.0 ** np.array(g) - 1, discounts[:len(g)]) for g in ideal])
    n_relevant = np.array([sum(v > 0 for v in rel.values()) for rel in grades])
    has_exact = np.array([2 in rel.values() for rel in grades])
    exact = values[:, :10] == 2
    return {"nDCG@10": ((2.0 ** values[:, :10] - 1) * discounts).sum(1) / idcg,
            "MRR@10": np.where(has_exact, np.where(exact.any(1), 1 / (exact.argmax(1) + 1), 0), np.nan),
            "Recall@10": (values[:, :10] > 0).sum(1) / n_relevant,
            "Recall@100": (values > 0).sum(1) / n_relevant}

## 3. BM25 with fixed field weights

Each field keeps its own IDF and length normalization. Final score:
`BM25_name + 0.5 × BM25_category + 0.25 × BM25_description`.
This is a sum of field scores, not BM25F. The weights are used directly; there is no parameter search.

In [3]:
porter = PorterStemmer()

@lru_cache(maxsize=200_000)
def stem(word):
    return porter.stem(word)


def tokenize(text):
    return [stem(t) for t in re.findall(r"\w+", text.lower())]


indexes = {}
for field in FIELDS:
    index = bm25s.BM25(method="lucene", k1=1.2, b=0.75)
    index.index([tokenize(text) for text in products[field]], show_progress=False)
    indexes[field] = index

query_tokens = [tokenize(text) for text in queries["query"].astype(str)]
bm25_ranked = []
for tokens in query_tokens:
    scores = np.zeros(len(products), dtype=np.float32)
    if tokens:
        for field in FIELDS:
            scores += WEIGHTS[field] * indexes[field].get_scores(tokens)
    bm25_ranked.append(topk(scores, positive_only=True))
rankings = {"BM25 (fixed weights)": np.stack(bm25_ranked)}

## 4. GTE-small on the concatenated text

Input order: **name → category → description**. One normalized 384-dimensional vector represents each
product. GTE-small's default input limit is 512 tokens; report truncation so its visible text is clear.
Cosine scores are normalized-vector dot products against every product, in batches of 16 queries.

In [4]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
if model.device.type == "cpu":
    model.float()  # The GTE checkpoint is FP16; FP32 is much faster on this CPU.
product_texts = products[FIELDS].agg(" ".join, axis=1).tolist()
query_texts = queries["query"].astype(str).tolist()

def truncated_pct(texts):
    truncated = 0
    for start in range(0, len(texts), 256):
        tokens = model.tokenizer(texts[start:start + 256], truncation=False, padding=False, verbose=False)
        truncated += sum(len(ids) > model.max_seq_length for ids in tokens["input_ids"])
    return 100 * truncated / len(texts)


print(f"Input limit: {model.max_seq_length} tokens; "
      f"truncated products: {truncated_pct(product_texts):.2f}%; "
      f"truncated queries: {truncated_pct(query_texts):.2f}%")
product_vectors = model.encode(product_texts, batch_size=64, normalize_embeddings=True,
                               convert_to_numpy=True, show_progress_bar=False)
query_vectors = model.encode(query_texts, batch_size=64, normalize_embeddings=True,
                             convert_to_numpy=True, show_progress_bar=False)
assert product_vectors.shape == (len(products), 384)
assert query_vectors.shape == (len(queries), 384)
assert np.allclose(np.linalg.norm(product_vectors, axis=1), 1, atol=1e-4)
assert np.allclose(np.linalg.norm(query_vectors, axis=1), 1, atol=1e-4)
gte_ranked = []
for start in range(0, len(query_vectors), 16):
    scores = query_vectors[start:start + 16] @ product_vectors.T
    gte_ranked.extend(topk(row) for row in scores)
rankings["GTE-small"] = np.stack(gte_ranked)

Input limit: 512 tokens; truncated products: 0.08%; truncated queries: 0.00%


## 5. Results and query examples

The table compares the same queries for both systems. Wins/losses use per-query nDCG@10, not raw
BM25 and cosine scores, whose scales are different. Inspect two fixed examples and one largest
win/loss; the extremes are illustrative rather than representative of all queries.

In [5]:
metrics = {name: evaluate(ranked) for name, ranked in rankings.items()}
summary = pd.DataFrame({name: {m: np.nanmean(v) for m, v in values.items()}
                        for name, values in metrics.items()}).T
display(summary.round(4))
delta = metrics["GTE-small"]["nDCG@10"] - metrics["BM25 (fixed weights)"]["nDCG@10"]
print(f"GTE nDCG difference: {100 * delta.mean():+.2f} points; "
      f"wins: {(delta > 1e-10).sum()}, losses: {(delta < -1e-10).sum()}, "
      f"ties: {(np.abs(delta) <= 1e-10).sum()}")

example_ids = np.flatnonzero(queries["query"].isin(["bathroom vanity knobs", "glass doors for bath"])).tolist()
if (delta > 1e-10).any():
    example_ids.append(int(np.argmax(delta)))
if (delta < -1e-10).any():
    example_ids.append(int(np.argmin(delta)))
for qi in dict.fromkeys(example_ids):
    print(f"\n{queries.iloc[qi]['query']} | Δ nDCG@10 {delta[qi]:+.3f}")
    rows = []
    for name, ranked in rankings.items():
        for rank, d in enumerate(ranked[qi, :3], 1):
            if d >= 0:
                rows.append({"method": name, "rank": rank, "product": products.iloc[d].product_name,
                             "label": {0: "Irrelevant", 1: "Partial", 2: "Exact"}.get(grades[qi].get(int(d)), "Unjudged")})
    display(pd.DataFrame(rows))

GTE nDCG difference: +0.19 points; wins: 169, losses: 189, ties: 121

bathroom vanity knobs | Δ nDCG@10 +0.073

glass doors for bath | Δ nDCG@10 +0.844

donaldson teak couch | Δ nDCG@10 +1.000

rattan truck | Δ nDCG@10 -0.922


,nDCG@10,MRR@10,Recall@10,Recall@100
BM25 (fixed weights),0.7158,0.6887,0.0640,0.4098
GTE-small,0.7177,0.7151,0.0609,0.3801


,method,rank,product,label
0,BM25 (fixed weights),1,42 '' single bathroom vanity,Irrelevant
1,BM25 (fixed weights),2,32 '' single bathroom vanity set,Irrelevant
2,BM25 (fixed weights),3,watqen vanities 24 '' single bathroom vanity set,Irrelevant
3,GTE-small,1,franzen 55 '' single bathroom vanity set,Irrelevant
4,GTE-small,2,fawkes 36 '' single bathroom vanity set,Irrelevant
5,GTE-small,3,freeland 37 '' single bathroom vanity set,Irrelevant


,method,rank,product,label
0,BM25 (fixed weights),1,glass wood french doors,Irrelevant
1,BM25 (fixed weights),2,quadro glass french doors,Irrelevant
2,BM25 (fixed weights),3,lucia glass french doors,Irrelevant
3,GTE-small,1,fluence 59.63 '' x 58.31 '' bypass bath door,Exact
4,GTE-small,2,fluence 59.63 '' x 58.31 '' bypass bath door w...,Exact
5,GTE-small,3,glass 60 '' w x 72 '' h bypass framed shower door,Exact


,method,rank,product,label
0,BM25 (fixed weights),1,teak coffee table,Unjudged
1,BM25 (fixed weights),2,extra large and wide couch riser,Unjudged
2,BM25 (fixed weights),3,shutt teak end table,Unjudged
3,GTE-small,1,summerton teak patio sofa with cushions,Partial
4,GTE-small,2,cambridge teak patio sofa with cushions,Partial
5,GTE-small,3,brunswick 5 piece teak sofa seating group with...,Partial


,method,rank,product,label
0,BM25 (fixed weights),1,2 piece truck dolly set,Partial
1,BM25 (fixed weights),2,500 lbs . capacity aluminum hand truck dolly,Partial
2,BM25 (fixed weights),3,hand truck dolly,Partial
3,GTE-small,1,alaze 4 piece outdoor furniture complete patio...,Unjudged
4,GTE-small,2,gutha outdoor patio furniture 3 piece rattan s...,Unjudged
5,GTE-small,3,siewert 31 '' wide papasan chair,Unjudged
